## Agenda

- **0. Real-world harms.** Why this is not abstract.
- **1. What a deepfake is, mechanically.**
- **2. The neural-network machinery.**
- **3. Evolution.** 2017 to 2026. One table.
- **4. LivePortrait.** Video-driven reenactment. Verified commands.
- **5. SimSwap and FaceFusion.** Face swap.
- **6. InstantID and DreamID.** The diffusion era.
- **7. GAN era vs. diffusion era.** Side by side.
- **8. Consent and the law.** Not optional.
- **9. Toolbox, platform notes, and takeaways.**

This notebook is designed as a keynote plus guided lab. Keep the audience in the conceptual material while the Mac runs the expensive cells.

| Time | Segment | Teaching move |
|---|---|---|
| 0-10 min | Opening and harms | Establish consent, provenance, and why the topic is not abstract. |
| 10-25 min | Mechanical pipeline | Ask the room to predict where extraction, reconstruction, and compositing fail. |
| 25-40 min | Neural machinery | Use the bottleneck diagrams to connect pixels, features, latent space, and identity. |
| 40-50 min | Evolution | Pause at each constraint removed: pair training, identity training, then source warping. |
| 50-75 min | LivePortrait lab | Run the three variants. Ask participants to compare motion, stitching, and artifacts. |
| 75-85 min | Break and output review | Let the videos render; discuss what is visibly synthetic and what is not. |
| 85-105 min | SimSwap, FaceFusion, InstantID, PhotoMaker | Separate architecture, tool availability, and actual Mac runtime status. |
| 105-120 min | Detection and provenance | Connect artifacts to the companion detection material, C2PA, and labeling. |
| 120-135 min | Consent and law | Use the excluded-image decisions as a practical ethics case study. |
| 135-150 min | Q&A and takeaways | Return to the throughline: consent, intent, provenance, and detection. |

### Audience checkpoints

- **Before the first model:** Ask, "What information must a model preserve for a face to remain recognizable?"
- **Before LivePortrait:** Ask participants to predict which variant will produce the strongest artifacts.
- **After LivePortrait:** Compare source, driving clip, and generated output. Ask which artifacts are temporal and which are spatial.
- **Before diffusion methods:** Ask why identity embeddings can be more flexible than directly warping source pixels.
- **Before Section 8:** Ask whether academic publication makes the use of a real person's likeness acceptable.
- **At the end:** Have each participant state one consent rule and one provenance practice they will use.

### Presenter safety rules

- Use only the bundled presenter portrait, your own media, or media with explicit written consent.
- Do not substitute celebrity, political, colleague, or audience images during a live demo.
- Label every generated output as synthetic and preserve the source, model, prompt, and parameter record.
- Keep the downloaded model repositories and generated videos local; the repository's `.gitignore` excludes them from GitHub.

## 0. The Real-World Harms

Four harm categories. Every number below has a named source.

**AI-generated CSAM.** Scale is growing fast. Enforcement is racing to catch up.

- NCMEC, 2026 report on 2025 data:
  - Over 400,000 CyberTipline reports in 2025 involved a generative-AI nexus.
  - Over 182,000 involved an offender possessing, generating, or attempting to generate AI-CSAM.
  - About 158,000 files have been categorized as AI-generated CSAM since January 2023.
- IWF (UK), 2026 report, "Harm Without Limits":
  - AI-generated CSAM videos assessed as illegal rose from 13 in 2024 to 3,443 in 2025.
  - That is roughly a 260-times increase in one year.
  - 65% of 2025 AI videos were rated the most severe UK legal category.
  - 97% of victims depicted are girls.
- Policy response: prosecuted today under the existing US PROTECT Act framework. The first federal
  AI-CSAM conviction was May 2024. The STOP CSAM Act is still pending in Congress as of 2026.
- Scope note: scale and policy only. No methods. No imagery. This matches what NCMEC and IWF publish
  themselves.

**Elder-targeted voice-clone scams.** The grandparent scam, industrialized.

- FBI IC3, 2024 Elder Fraud Report:
  - Adults 60 and older lost $4.885 billion. That is 147,127 complaints.
  - Losses rose 43% year over year. Complaints rose 46%.
  - 7,500 victims each lost over $100,000. Average loss was $83,000.
- FBI IC3, 2025 Annual Report:
  - First-ever dedicated AI-fraud category. 22,364 complaints. $893 million in AI-related losses.
  - $352 million of that was lost by victims 60 and older.
- AARP, 2025 Fraud Survey: 84% of adults 50 and older are concerned about AI voice-cloning scams. 51%
  say very concerned.

  ![FBI IC3: top reported elder fraud types, 2023](images/01_fbi_ic3_2024_elder_fraud_cropped.png)

  The FBI's own chart. AI and voice-cloning fraud is not yet its own category here. Tech support scams
  still lead by complaint count. That is why the IC3's new AI-fraud category, above, is the number to
  watch going forward.

  ![ABA Foundation + FBI: Deepfake Media Scams infographic](images/02_aba_fbi_deepfake_scams.jpg)

  Official FBI-co-branded infographic. ABA Foundation, September 2025. Detection tips and red flags for
  deepfake scams. Its headline number, $50.5 billion lost since 2020, covers every fraud type combined.
  It is not AI-specific. The AI-specific number is above.

**Non-consensual intimate imagery.**

- Home Security Heroes, 2023: about 98% of deepfake video content online is non-consensual pornography.
  99% of subjects are women.
- This is 2023 data. It is the number most commonly cited. It is also dated. Say so if asked.

**Political and election disinformation.**

- January 2024. An AI-cloned Biden voice robocall told New Hampshire primary voters not to vote.
- The FCC fined political consultant Steve Kramer $6 million in September 2024. He was also criminally
  indicted in New Hampshire.
- The best-known *consented* example of this exact technique is the 2018 Jordan Peele and Obama PSA,
  made with BuzzFeed. It combines a face swap with a voice impression. It was built specifically as a
  media-literacy demonstration, with full public knowledge it was synthetic. That is the difference
  between this and the robocall case above. Worth citing on stage if the audience asks "what does one
  of these actually look like."

**Throughline.** Every method in this notebook is dual-use. The only thing separating a workshop demo
from one of the four harms above is consent and intent. Section 8 is not optional.

In [ ]:
# Two original charts. Not sourced graphics.
# IWF's report shows the trend in text and table form only.
# The FBI's 2025 AI-fraud figures live inside a PDF with no standalone chart.
# Both charts below are built from the verified numbers.
# Same visual system as the companion detection notebook.
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
%matplotlib inline

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
MAGENTA, GREEN, VIOLET, RED = "#e87ba4", "#008300", "#4a3aa7", "#e34948"
INK, INK_SECONDARY, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.family": "sans-serif", "font.size": 11,
})

def style_axes(ax, value_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    (ax.xaxis if value_axis == "x" else ax.yaxis).grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    return ax

# Chart A. IWF AI-generated CSAM videos assessed illegal. 2024 vs 2025. Log scale.
# The raw jump is a ~260x increase. It would make the 2024 bar invisible on a linear axis.
fig, ax = plt.subplots(figsize=(6, 4.5))
years, counts = ["2024", "2025"], [13, 3443]
bars = ax.bar(years, counts, color=BLUE, width=0.5, zorder=3)
ax.set_yscale("log")
ax.set_ylabel("AI-generated CSAM videos assessed illegal (log scale)")
ax.set_title("IWF: AI-generated CSAM videos, one year apart", loc="left", fontsize=12, fontweight="bold")
style_axes(ax, value_axis="y")
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, val * 1.15, f"{val:,}", ha="center", fontsize=11, color=INK)
ax.text(0.98, 0.92, "~260x increase", transform=ax.transAxes, ha="right", fontsize=10,
        color=INK_SECONDARY, fontweight="bold")
fig.text(0.01, -0.05, "Source: Internet Watch Foundation, \"Harm Without Limits\" (2026).", fontsize=8, color=MUTED)
plt.tight_layout(); plt.show()

# Chart B. FBI IC3 2025 AI-fraud losses. Total vs. seniors-attributed. Same report, same year.
fig, ax = plt.subplots(figsize=(6, 4.5))
labels, values = ["All AI-fraud losses\n(2025)", "Attributed to\nvictims 60+"], [893, 352]
bars = ax.bar(labels, values, color=[BLUE, ORANGE], width=0.5, zorder=3)
ax.set_ylabel("Losses ($ millions)")
ax.set_title("FBI IC3: AI-fraud losses, 2025. The share hitting seniors.",
             loc="left", fontsize=12, fontweight="bold")
style_axes(ax, value_axis="y")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 15, f"${val}M", ha="center", fontsize=11, color=INK)
fig.text(0.01, -0.05, "Source: FBI, 2025 IC3 Annual Report (via FBI press release, 2026).", fontsize=8, color=MUTED)
plt.tight_layout(); plt.show()

In [ ]:
# Utility for the "see different outputs" cells in Sections 4-6.
# No fabricated visuals. A side-by-side viewer for files you generate.
import os
from PIL import Image

def show_outputs_grid(paths, labels, cell_width=4, cell_height=4):
    available = [(p, l) for p, l in zip(paths, labels) if os.path.exists(p)]
    if not available:
        print("No output files found yet. Run the variant commands in this section first.")
        print("Then re-run this cell to compare them side by side.")
        return
    fig, axes = plt.subplots(1, len(available), figsize=(cell_width * len(available), cell_height))
    if len(available) == 1:
        axes = [axes]
    for ax, (path, label) in zip(axes, available):
        ax.imshow(Image.open(path))
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("show_outputs_grid() ready.")

## 1. What Is a Deepfake, Mechanically?

Three steps. Every pipeline in this notebook. No exceptions.

**Extraction.** Detect the face in each frame. Align it to a standard pose and size. Commonly 256x256.
Nearly every downstream model assumes this consistent input.

![Extraction: face detection and alignment](images/01_extraction_face_detection_alignment.png)

**Training.** Learn to reconstruct a face from a shared, identity-agnostic representation. Classic
version: two decoders share one encoder. Decode identity A's features with identity B's decoder. That
is the swap.

![Training: cross-reconstruction between two identities](images/02_training_cross_reconstruction.png)

**Creation.** Run it on every frame. Extract. Reconstruct with the swapped decoder. Merge back into the
original frame. Colour-correct.

![Creation: merging the reconstructed face back into the frame](images/03_creation_merge_frame.png)
![Creation: colour correction](images/04_creation_color_correction.png)

**Where this breaks, historically.** The merge and colour-correction step is hard-coded. Not learned. No
cross-frame consistency check. That is why older deepfakes flicker and show visible seams.

## 2. The Neural-Network Machinery

Every method here, 2017 to 2026, rests on two ideas.

**Idea one.** A network that turns pixels into features.

![A basic feed-forward neural network](images/05_neural_network_layers.png)

**Why convolutional networks work for faces.** Layers represent progressively more complex features.
Edges. Then parts. Then identity.

![Progressively complex CNN feature layers](images/06_cnn_feature_hierarchy.png)

**Idea two.** A network shaped to force useful compression. Why an autoencoder specifically? It only
works if forced to compress. No bottleneck. The network cheats. It wires input straight to output.
Learns nothing useful.

![Autoencoder without a bottleneck](images/07_autoencoder_degenerate.png)

**Force the bottleneck.** Now the network must learn a compact, reusable representation. That reusable
latent space is what a face-swap model exploits. One encoder for face structure. Different decoders per
identity.

![A proper encoder/decoder autoencoder](images/08_autoencoder_encoder_decoder.png)

## 3. The Evolution

| Era | Method | What changed |
|---|---|---|
| 2017 | Reddit r/deepfakes autoencoder apps | Shared-encoder swap. Hours to days of training per pair |
| Dec 2019 | First Order Motion Model (NeurIPS 2019) | Animate any source with any driving video. Zero per-identity training |
| Jul 2020 | SimSwap (ACM MM 2020) | One-shot, arbitrary-identity face swap. No per-pair training |
| Mar 2022 | Thin-Plate-Spline Motion Model (CVPR 2022) | Better warping for larger pose gaps than FOMM |
| Jan 2024 | InstantID | Identity as a diffusion conditioning signal. No source-image warping at all |
| Jul 2024 | LivePortrait (Kuaishou) | Real-time. ~69M-frame training. Stitching and retargeting control |
| Apr 2025 | DreamID (ByteDance, SIGGRAPH Asia 2025) | Single-step diffusion face swap. In production (Dreamina) |
| Jan 2026 | DreamID-V (ECCV 2026 Oral) | Video follow-up to DreamID |

**Throughline.** Each era removed a constraint. First per-pair training. Then per-identity training.
Then the source image itself as something to warp.

## 4. Video-Driven Reenactment: LivePortrait

- One source portrait plus one driving video. Source animated to follow the video's motion.
- No per-identity training. Kuaishou, arXiv:2407.03168.
- What changed vs. 2019/2022: ~69M-frame training set.
- Explicit stitching and retargeting control. Fixes where FOMM/TPS most visibly broke. Shoulder and
  background attachment. Eye and mouth independence from head pose.
- Close to real-time on one consumer GPU.

![LivePortrait architecture](images/01_liveportrait_architecture.png)

**It generalizes beyond photographs.** The paper's own qualitative figure: paintings, a marble statue, a
vintage photo, a cartoon illustration. All animated by the same model.

![LivePortrait qualitative results](images/05_liveportrait_qualitative_results.png)

In [ ]:
# LivePortrait setup for this macOS Apple Silicon environment.
# The repository is already downloaded to LivePortrait/.
#
# Install the macOS requirements from Terminal while the aipr environment is active:
#   python -m pip install -r LivePortrait/requirements_macOS.txt
#   conda install -n aipr -c conda-forge ffmpeg -y
#
# Download the official pretrained weights once:
#   python -m pip install "huggingface_hub==1.33.0"
#   python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='KlingTeam/LivePortrait', local_dir='LivePortrait/pretrained_weights', ignore_patterns=['*.git*', 'README.md', 'docs/**'])"
#
# The runnable cell below uses the repository's bundled, public example clip:
#   LivePortrait/assets/examples/driving/d0.mp4
#
# Apple Silicon note: PYTORCH_ENABLE_MPS_FALLBACK=1 is set for unsupported MPS operators.
print("LivePortrait macOS setup is documented. Run the next cell for the three variants.")

In [ ]:
# LivePortrait: download, verify inputs, and run three consented variants.
# The bundled driving clip is an official repository example.
from pathlib import Path
import os
import subprocess

workspace_dir = Path.cwd()
liveportrait_dir = workspace_dir / "LivePortrait"
source_image = workspace_dir / "images/ghulam-mujitaba-480x480.jpg"
driving_video = liveportrait_dir / "assets/examples/driving/d0.mp4"

if not liveportrait_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/KlingTeam/LivePortrait", str(liveportrait_dir)],
        check=True,
    )

if not source_image.exists():
    print(f"Source image not found: {source_image}")
    commands = []
elif not driving_video.exists():
    print(f"Driving video not found: {driving_video}")
    commands = []
else:
    commands = [
        ["python", "inference.py", "-s", str(source_image), "-d", str(driving_video), "-o", "out_relative"],
        ["python", "inference.py", "-s", str(source_image), "-d", str(driving_video), "-o", "out_absolute", "--no_flag_relative_motion"],
        ["python", "inference.py", "-s", str(source_image), "-d", str(driving_video), "-o", "out_amplified", "--no_flag_stitching", "--driving_multiplier", "1.75"],
    ]

run_environment = os.environ.copy()
run_environment["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

for command in commands:
    subprocess.run(command, cwd=liveportrait_dir, env=run_environment, check=True)

if commands:
    print("LivePortrait variants completed. Run the comparison cell above to view generated frames.")
else:
    print("LivePortrait setup is ready; inference was skipped until the required inputs are available.")

In [ ]:
# Display the generated LivePortrait videos.
from pathlib import Path
from IPython.display import Video, display

output_videos = sorted(Path("LivePortrait").glob("out_*/*.mp4"))
if not output_videos:
    print("No LivePortrait videos found. Run the inference cell first.")
else:
    for output_video in output_videos:
        print(output_video)
        display(Video(str(output_video), embed=False))

## Results Gallery and Audience Interpretation

This is the reproducible result from the supported Apple Silicon path. The same presenter portrait and the official bundled driving clip are passed through three LivePortrait configurations.

- **Relative motion:** preserves the driving video's relative movement and is the best baseline for a first explanation.
- **Absolute motion:** transfers absolute pose information. It can look more surprising when the source and driving subjects differ in framing.
- **Amplified motion without stitching:** increases motion and removes the stitching correction. It is useful for showing how a quality-control choice becomes a visible artifact.

When reviewing the videos, ask the audience to look for four things:

1. **Temporal consistency:** does the identity remain stable from frame to frame?
2. **Boundary behavior:** does the generated face attach naturally to hair, neck, shoulders, and background?
3. **Motion transfer:** which expressions and head movements came from the driving clip?
4. **Provenance:** can a viewer tell which media and parameters produced the result?

The other methods are not silently represented as successful Mac outputs. InstantID is CUDA/SDXL-oriented and impractical on this Mac; PhotoMaker is prepared as the MPS fallback but needs its model download and full generation run; FaceFusion, SimSwap, and DreamID require separate checkpoints or runtimes. That distinction is part of the technical lesson.

## 5. Face Swap: SimSwap and FaceFusion

**SimSwap.** arXiv:2106.06340. The GAN architecture underneath most one-shot swap tools.

- Encoder reads the target.
- An ID Injection Module blends in the source identity.
- Decoder renders the swap.
- No retraining per identity pair.

![SimSwap architecture](images/02_simswap_architecture.png)

Not reproduced here: SimSwap's own qualitative-results grid. It demonstrates the method entirely on
real, named celebrities as face-swap sources. Excluded regardless of academic-citation context. See
Section 8.

**FaceFusion.** `facefusion/facefusion`. No paper. Currently v3.9.0.

- The actively-maintained toolkit most people run today.
- Swap model, plus detection and alignment, plus an enhancer. One CLI/GUI pipeline.
- A real swap needs two different identities. Use your own photo for one. Ask a colleague or friend for the other, or use a licensed stock photo. Not a celebrity's.

In [ ]:
# FaceFusion setup.  pip install facefusion
#
# Honesty note: exact -s/-t/-o path-flag syntax for headless mode was NOT confirmed from primary
# docs. Run this once to get the real syntax before your talk:
#   python facefusion.py headless-run --help
#
# The PROCESSOR flags below ARE verified against docs.facefusion.io.
# Safe to append to whatever your verified headless-run invocation turns out to be.

variant_flags = {
    "docs default":
        "--processors face_swapper face_enhancer --face-swapper-model inswapper_128 "
        "--face-swapper-weight 1.0 --face-enhancer-model gfpgan_1.4 --face-enhancer-blend 80",
    "lighter swap, subtler enhance":
        "--processors face_swapper face_enhancer --face-swapper-model hyperswap_1a_256 "
        "--face-swapper-weight 0.5 --face-enhancer-model codeformer --face-enhancer-blend 40",
    "high-res swap, max enhance":
        "--processors face_swapper face_enhancer --face-swapper-model hyperswap_1a_256 "
        "--face-swapper-pixel-boost 512x512 --face-enhancer-model gpen_bfr_1024 --face-enhancer-blend 100",
}
for label, flags in variant_flags.items():
    print(f"[{label}]\n  <your verified headless-run -s/-t/-o command> {flags}\n")

# Speaker: source and target media must be your own. Or media you have explicit written consent
# to use. Leave FaceFusion's built-in content-safety check on for a live demo.

## 6. The Diffusion-Era Shift: InstantID and DreamID

GAN methods warp or blend pixels from a specific source image. FOMM. TPS. SimSwap. LivePortrait. All of
them.

2024 to 2026: a shift. Identity becomes a conditioning signal. Injected into diffusion generation
instead.

**InstantID.** arXiv:2401.07519.

- Face embedding, plus a decoupled cross-attention IP-Adapter, plus a ControlNet-style IdentityNet.
- Tuning-free identity generation. Single forward pass.

![InstantID architecture](images/03_instantid_architecture.png)

Not reproduced here: InstantID's qualitative figure. Source-photo identities could not be confidently
cleared. See Section 8.

**DreamID.** ByteDance. arXiv:2504.14509. SIGGRAPH Asia 2025.

- Applies the same idea to face-swapping.
- SwapNet, a base UNet. FaceNet, a pixel-level identity encoder. An ID Adapter.
- Trained with Triplet-ID-Group supervision.
- Single-step diffusion face-swap. Deployed in production, Dreamina.
- A video follow-up, DreamID-V, was an ECCV 2026 Oral.

![DreamID architecture](images/04_dreamid_architecture.png)

![DreamID qualitative results on FFHQ](images/08_dreamid_qualitative_results_ffhq_only.png)

Figure 5, cropped to its FFHQ-dataset rows only. DreamID vs. eight other methods. Inswapper. SimSwap.
CSCS. FaceDancer. DiffFace. DiffSwap. FaceAdapter. REFace. All on anonymous research faces. The paper's
web-data rows are excluded. Provenance and consent were unconfirmed.

In [ ]:
# InstantID setup and results audit.
# InstantID is isolated in the `instantid` environment; do not replace the working aipr stack.
#
# The project and model assets are downloaded by the README commands:
#   conda activate instantid
#   git clone https://github.com/instantX-research/InstantID
#   python -m pip install diffusers==0.25.1 torch==2.0.0 torchvision==0.15.1 transformers==4.37.1 accelerate safetensors einops omegaconf peft opencv-python insightface gradio controlnet_aux gdown==5.1.0 onnxruntime
#   cd InstantID && python gradio_demo/download_models.py
#
# The upstream infer.py assumes CUDA. The adapted runner `instantid_mps_demo.py`
# uses MPS on Apple Silicon and writes `instantid_result.png` in this workspace.
from pathlib import Path
from IPython.display import Image, display

instantid_dir = Path("InstantID")
checkpoint_dir = instantid_dir / "checkpoints"
face_model_dir = instantid_dir / "models" / "antelopev2"
result_path = Path("instantid_result.png")

print(f"InstantID folder: {instantid_dir.resolve()}")
print(f"Repository present: {instantid_dir.exists()}")
print(f"Adapter present: {(checkpoint_dir / 'ip-adapter.bin').exists()}")
print(f"ControlNet present: {(checkpoint_dir / 'ControlNetModel' / 'diffusion_pytorch_model.safetensors').exists()}")
print(f"antelopev2 present: {face_model_dir.exists()}")
print(f"Result present: {result_path.exists()}")

if result_path.exists():
    display(Image(filename=str(result_path)))
else:
    print("Run from Terminal: PYTHONPATH=InstantID conda run -n instantid python instantid_mps_demo.py")

### Runtime status for this notebook

- **LivePortrait:** runnable on this Mac. The notebook downloads weights, uses the bundled `d0.mp4`, and runs three variants.
- **InstantID:** the project and checkpoints are downloaded, but its upstream SDXL/CUDA assumptions make generation impractical on Apple Silicon. Keep it as a GPU/Colab comparison.
- **PhotoMaker:** repository downloaded and imports verified in the separate `photomaker` environment with MPS available. It is the recommended macOS fallback for identity-preserving image generation.
- **FaceFusion:** repository downloaded, but install and media processing remain separate because its CLI and processor models must be verified with consented inputs.
- **SimSwap:** repository downloaded, but its official workflow requires separate pretrained weights and is primarily GPU-oriented.
- **DreamID:** repository downloaded, but it contains project/provenance material without a local executable inference pipeline and requires external model assets.

In [ ]:
# PhotoMaker: macOS-oriented alternative to InstantID.
# Project: https://github.com/TencentARC/PhotoMaker
# Install in a separate environment from Terminal:
#   conda create -n photomaker python=3.10 -y
#   conda activate photomaker
#   python -m pip install --upgrade pip
#   python -m pip install torch==2.1.2
#   python -m pip install -r PhotoMaker/requirements.txt
#   python -m pip install git+https://github.com/TencentARC/PhotoMaker.git
#
# The official Mac notes also list:
#   xcode-select --install
#   brew install llvm libomp

from pathlib import Path

photomaker_dir = Path("PhotoMaker")
example_dir = photomaker_dir / "examples"
print(f"PhotoMaker folder: {photomaker_dir.resolve()}")
print(f"Repository present: {photomaker_dir.exists()}")
print(f"Example identity folders: {sorted(path.name for path in example_dir.iterdir() if path.is_dir()) if example_dir.exists() else []}")
print("PhotoMaker is the recommended macOS fallback when InstantID generation is unavailable.")

In [ ]:
# Audit the remaining method repositories.
from pathlib import Path

for project_name in ("FaceFusion", "SimSwap", "DreamID"):
    project_dir = Path(project_name)
    print(f"{project_name}: {'downloaded' if project_dir.exists() else 'missing'}")

print("FaceFusion requires a verified CLI install and consented source/target media.")
print("SimSwap requires its separate pretrained weights before inference.")
print("DreamID has no local executable inference entry point in this checkout.")

## 7. GAN Era vs. Diffusion Era, Side by Side

| | FOMM / TPS / LivePortrait | SimSwap / FaceFusion | InstantID / DreamID |
|---|---|---|---|
| **Needs** | source image, driving video | source face, target image or video | one or a few reference photos |
| **Per-identity training** | none | none | none |
| **Mechanism** | keypoint/warp-field motion transfer | encoder to ID injection to decoder | identity embedding into diffusion denoising |
| **Speed (2026)** | near real-time (LivePortrait) | real-time-ish (FaceFusion) | InstantID: seconds. DreamID: single-step |
| **Output** | video (reenactment) | image or video (swap) | image (InstantID). Image or video (DreamID) |
| **Verified variant knob** | `--flag_relative_motion`, `--driving_multiplier` | `--face-swapper-weight`, `--face-swapper-model` | `controlnet_conditioning_scale`, `ip_adapter_scale` |

**Detection throughline.** Each leaves a different artifact signature. Warping residuals. GAN spectral
fingerprints. Or diffusion reconstruction error. That is why one detector does not generalize across all
three. See the companion detection notebook, Section 7, Deepfake-Eval-2024.

## 8. Consent and the Law: Why This Isn't Optional

Section 0 was not throat-clearing. Generating a synthetic likeness of an identifiable real person
without consent sits on the same spectrum as the harms just covered. Sometimes it is the same statute.

**Four real decisions made while building this notebook.**

- Two face-swap images, from an earlier teaching-notebook source, showed a real senator composited with
  another real public figure's likeness. No consent. Excluded.
- In their place: a text citation to the 2018 Jordan Peele and Obama PSA, made with BuzzFeed. The
  standard, consented example of this exact technique, built specifically for media-literacy awareness.
  See Section 0.
- SimSwap's own published qualitative-results figure. Real, named celebrities as swap sources. Excluded
  from Section 5. Reproducing a research figure for citation is a defensible academic norm. This
  notebook's policy excludes it anyway.
- InstantID's qualitative figure. Identity could not be confidently cleared. Excluded from Section 6.
  DreamID's was cropped to its anonymous-dataset rows only.

**The law.**

- US TAKE IT DOWN Act, May 19, 2025. Platforms must remove non-consensual intimate imagery, AI-generated
  included. Criminalizes publishing it.
- EU AI Act, Article 50, applicable August 2, 2026. AI content must be machine-readably marked at
  creation. Deployers must disclose deepfakes to viewers.
- Standalone state and national deepfake-consent statutes increasingly cover political, commercial, and
  intimate-imagery cases separately.
- STOP CSAM Act. Still pending in US Congress as of 2026. See Section 0.

**Provenance is the responsible-use pattern.** Pair any pipeline here with C2PA Content Credentials or
Google SynthID at export. See the companion detection notebook, Section 8.

**One rule for this talk.** Your own face. A colleague's, with explicit sign-off. Or a licensed
synthetic or stock face. Never a public figure's likeness. Not for a demo. Not because a paper did it
first.

## 9. Toolbox, Platform Notes, and Takeaways

### Toolbox

| Method | Type | Repo | Paper |
|---|---|---|---|
| LivePortrait | Video-driven reenactment | `KwaiVGI/LivePortrait` | arXiv:2407.03168 |
| First Order Motion Model | Reenactment, 2019 baseline | `AliaksandrSiarohin/first-order-model` | NeurIPS 2019 |
| Thin-Plate-Spline Motion Model | Reenactment, 2022 baseline | `yoyo-nb/Thin-Plate-Spline-Motion-Model` | CVPR 2022 |
| SimSwap | Face swap (GAN) | `neuralchen/SimSwap` | arXiv:2106.06340 |
| FaceFusion | Face swap (toolkit) | `facefusion/facefusion` | none |
| InstantID | Identity-preserving diffusion | `instantX-research/InstantID` | arXiv:2401.07519 |
| DreamID | Diffusion face swap, single-step | `superhero-7/DreamID` | arXiv:2504.14509 |

All verified, currently-public repositories.

### Platform notes

The presenter is on macOS. Some of the audience may be on Windows.

- All shell commands in this notebook are written in macOS/Linux style.
- Windows, easiest path: use WSL (Windows Subsystem for Linux) or Git Bash. Commands work unchanged
  there.
- Windows PowerShell also works, with two adjustments:
  - Run any `&&`-chained command as two separate lines. Windows PowerShell 5.1 does not support `&&`.
  - Replace a trailing backslash line break with a backtick.
- `conda`, `pip`, `git`, and `python` commands are identical on every platform once installed.
- `ffmpeg`: `brew install ffmpeg` on macOS. `winget install ffmpeg` on Windows.

### Takeaways

- This is not an abstract problem. A 260-times jump in AI-CSAM video reports. $893 million in 2025
  AI-fraud losses. That is why this notebook opens with harms, not architecture.
- Every method here does extraction, then training, then creation, underneath. Sections 1 and 2 are the
  mechanism every later section specializes.
- Three paradigm shifts. One throughline. No per-pair training. Then no per-identity training. Then no
  source-image warping at all.
- "Different outputs" is a documented, verifiable parameter story. Not guesswork. For every method
  demoed here.
- Four real content decisions went into building this deck. That is the actual practice of consent. Not
  just the policy.